# Rate Experiment Analysis (photon-click rate + excitation-attempt rate)

Interactive analysis of `RateExperiment` runs (one row per heralded photon click).
Reuses the BLAS-free helpers from `analyze_rate.py`.

- **click rate**   = n_clicks / duration            [clicks/s]
- **attempt rate** = sum(trials_per_click) / duration [excitation attempts/s]
- **herald prob**  = n_clicks / total_attempts       [clicks per attempt, Wilson CI]
- **load rate**    = sum(how_many_atoms) / duration  [atom-load attempts/s]

Use section 4 (`compare`) to get the speed-up of a low-trap / OP-optimized run vs a
high-trap baseline run. Run with the working-BLAS `python` kernel (numpy / h5py / mpl).

In [ ]:
import sys, os, glob
from pathlib import Path
_ANALYSIS_DIR = r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\analysis"
if _ANALYSIS_DIR not in sys.path:
    sys.path.insert(0, _ANALYSIS_DIR)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2
from analyze_rate import RESULTS_DIR, find_run, load_run, summarize, compare, wilson
print("Results dir:", RESULTS_DIR)

## 1. List recent runs

In [ ]:
files = sorted(glob.glob(str(RESULTS_DIR / "**" / "*-RateExperiment.h5"), recursive=True),
               key=lambda f: int(os.path.basename(f).split("-")[0]))
if not files:
    print("No *-RateExperiment.h5 found yet.")
print(f"{'RID':>7} {'clicks':>7} {'dur(s)':>9} {'clicks/s':>9} {'attempts/s':>11} {'herald%':>8}")
print('-' * 56)
for f in files[-15:]:
    r = summarize(f, verbose=False)
    if r is None:
        rid = int(os.path.basename(f).split('-')[0])
        print(f"{rid:>7}   (no clicks)")
        continue
    print(f"{r['rid']:>7} {r['n_clicks']:>7} {r['duration']:>9.1f} "
          f"{r['clicks_per_s']:>9.3f} {r['attempts_per_s']:>11.1f} {100*r['herald_prob']:>8.3f}")

## 2. Select and summarize a run

Set `RID` to an int, or `None` for the latest.

In [ ]:
RID = None   # <-- int or None (latest)

path = find_run(RID)
print("File :", os.path.basename(path))
res = summarize(path)            # prints the headline rate numbers
data, meta, rid = load_run(path)

## 3. Plots — rate steadiness + attempts-per-click

In [ ]:
ts = np.sort(data["click_times_s"])
trials = data["trials_per_click"]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)

cum = np.arange(1, ts.size + 1)
axA.step(ts, cum, where="post", color="C0", lw=1.5)
axA.plot([0, res["duration"]], [0, res["clicks_per_s"] * res["duration"]], "--",
         color="C3", lw=1, label=f"mean {res['clicks_per_s']:.3f} clicks/s")
axA.set_xlabel("time since run start (s)"); axA.set_ylabel("cumulative clicks")
axA.set_title(f"Cumulative clicks vs time — RID {rid}")
axA.legend(loc="upper left"); axA.grid(alpha=0.3)

axB.hist(trials, bins=min(40, max(5, trials.size // 3)), color="C1",
         alpha=0.8, edgecolor="black", lw=0.4)
axB.axvline(float(np.mean(trials)), color="C3", ls="--", lw=1.2,
            label=f"mean {np.mean(trials):.1f} attempts/click")
axB.set_xlabel("excitation attempts between clicks"); axB.set_ylabel("count")
axB.set_title("Attempts per click  (1/herald-prob)")
axB.legend(loc="upper right"); axB.grid(alpha=0.3)
plt.show()

## 4. Compare baseline vs optimized

Set `RID_BEFORE` (high-trap baseline) and `RID_AFTER` (low-trap / OP-optimized) to print
the speed-up factors. herald prob should stay ~constant (the click physics is unchanged);
the speed-up shows up in clicks/s and attempts/s.

In [ ]:
RID_BEFORE = None   # <-- high-trap baseline RID
RID_AFTER  = None   # <-- low-trap / OP-optimized RID

if RID_BEFORE is not None and RID_AFTER is not None:
    compare(RID_BEFORE, RID_AFTER)
else:
    print("Set RID_BEFORE (baseline) and RID_AFTER (optimized) above to print the speed-up.")